In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
REPO_URL = "https://github.com/Jennt54321/fine-tune-gsm8k-socratic.git"  # edit for your fork

%cd /content
!rm -rf fine-tune-gsm8k-socratic
!git clone {REPO_URL}
%cd /content/fine-tune-gsm8k-socratic
!pwd

/content
Cloning into 'fine-tune-gsm8k-socratic'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 135 (delta 69), reused 116 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 2.21 MiB | 18.88 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/fine-tune-gsm8k-socratic
/content/fine-tune-gsm8k-socratic


In [ ]:
# Initialize LLaMA-Factory submodule
!git submodule update --init --recursive

Submodule 'LLaMA-Factory' (https://github.com/Jennt54321/LlamaFactory.git) registered for path 'LLaMA-Factory'
Cloning into '/content/fine-tune-gsm8k-socratic/LLaMA-Factory'...
Submodule path 'LLaMA-Factory': checked out 'dfecb84a7a7b06ad83e659cf3c6d1ed838ac4c6e'


In [ ]:
# Core deps (transformers, datasets, etc.)
!pip install -q -U peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


In [ ]:
!pip install -q -U transformers datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.7 MB/s eta 0:00:00


In [ ]:
!pip install rouge-chinese

In [ ]:
# Requirements (bert-score, etc. for evaluation)
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00


In [ ]:
# LLaMA-Factory (editable, with torch + bitsandbytes)
%cd /content/fine-tune-gsm8k-socratic/LLaMA-Factory
!pip install -e .[torch,bitsandbytes]
%cd /content/fine-tune-gsm8k-socratic

/content/fine-tune-gsm8k-socratic/LLaMA-Factory
Obtaining file:///content/fine-tune-gsm8k-socratic/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.0 MB/s eta 0:0

In [ ]:
# Verify llamafactory-cli is available
!llamafactory-cli version

----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.5.dev0           |
|                                                        |
| Project page: https://github.com/hiyouga/LLaMA-Factory |
----------------------------------------------------------


In [ ]:
# (If Drive is mounted) Create output dirs so training + eval outputs persist on Drive.
import os
DRIVE_BASE = "/content/drive/MyDrive/llm_outputs"
DRIVE_CHECKPOINT = f"{DRIVE_BASE}/socratic_qwen25-3b-instruct"
DRIVE_EVAL_PROMPTONLY = f"{DRIVE_BASE}/gsm8k_socratic_qwen_eval_promptonly"
DRIVE_EVAL_FINETUNED = f"{DRIVE_BASE}/gsm8k_socratic_qwen_eval_finetuned"

if os.path.isdir("/content/drive"):
    for d in [DRIVE_CHECKPOINT, DRIVE_EVAL_PROMPTONLY, DRIVE_EVAL_FINETUNED]:
        os.makedirs(d, exist_ok=True)
    print(f"Checkpoints: {DRIVE_CHECKPOINT}")
    print(f"Eval (prompt-only): {DRIVE_EVAL_PROMPTONLY}")
    print(f"Eval (fine-tuned): {DRIVE_EVAL_FINETUNED}")
else:
    print("Drive not mounted. Outputs will go to outputs/ (lost when runtime ends).")

Checkpoints: /content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct
Eval (prompt-only): /content/drive/MyDrive/llm_outputs/gsm8k_socratic_qwen_eval_promptonly
Eval (fine-tuned): /content/drive/MyDrive/llm_outputs/gsm8k_socratic_qwen_eval_finetuned


In [ ]:
%cd /content/fine-tune-gsm8k-socratic
!python scripts/prepare_socratic_dataset.py

/content/fine-tune-gsm8k-socratic
正在下載並準備 GSM8K socratic 資料集...
README.md: 7.93kB [00:00, 17.0MB/s]
socratic/train-00000-of-00001.parquet: 100% 2.68M/2.68M [00:01<00:00, 2.43MB/s]
socratic/test-00000-of-00001.parquet: 100% 487k/487k [00:00<00:00, 1.49MB/s]
Generating train split: 100% 7473/7473 [00:00<00:00, 225766.27 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 267428.19 examples/s]
Converting train: 100% 7473/7473 [00:00<00:00, 26406.12it/s]
Converting test: 100% 1319/1319 [00:00<00:00, 25345.84it/s]
✅ GSM8K socratic 資料準備完成並已註冊（gsm8k_socratic_train / gsm8k_socratic_test）。


In [ ]:
# 訓練 (full run)
!llamafactory-cli train socratic_train_config.yaml output_dir=/content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct
# max_samples 快速測試: !llamafactory-cli train socratic_train_config.yaml output_dir=/content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct max_samples=100

In [ ]:
# max_samples 快速測試: !llamafactory-cli train socratic_promptonly_eval.yaml output_dir=/content/drive/MyDrive/llm_outputs/gsm8k_socratic_qwen_eval_promptonly max_samples=100
!llamafactory-cli train socratic_promptonly_eval.yaml output_dir=/content/drive/MyDrive/llm_outputs/gsm8k_socratic_qwen_eval_promptonly

[INFO|2026-03-04 03:25:42] llamafactory.hparams.parser:508 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:670] 2026-03-04 03:25:43,185 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json
[INFO|configuration_utils.py:742] 2026-03-04 03:25:43,188 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "f

In [ ]:
# max_samples 快速測試: !llamafactory-cli train socratic_finetuned_eval.yaml output_dir=/content/drive/MyDrive/llm_outputs/gsm8k_socratic_qwen_eval_finetuned adapter_name_or_path=/content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct max_samples=100
!llamafactory-cli train socratic_finetuned_eval.yaml output_dir=/content/drive/MyDrive/llm_outputs/gsm8k_socratic_qwen_eval_finetuned adapter_name_or_path=/content/drive/MyDrive/llm_outputs/socratic_qwen25-3b-instruct

[INFO|2026-03-04 08:33:40] llamafactory.hparams.parser:508 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:670] 2026-03-04 08:33:41,185 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json
[INFO|configuration_utils.py:742] 2026-03-04 08:33:41,188 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "f